# Sesión 8 · Ajuste, comparación y guardado de modelos
**Curso:** Uso de IA para el análisis descriptivo y predictivo en energías renovables

> Ejecute cada celda con `Shift + Enter` y lea la explicación antes de cada paso. Todos los datos son **sintéticos** (simulados) para practicar.

**Objetivo:** buscar hiperparámetros con validación para series de tiempo, comparar modelos con varios criterios, explicarlos y guardar el mejor.

## 1. Datos y variables
Predecimos la potencia de la hora usando condiciones meteorológicas y la hora (como seno y coseno).

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
rng = np.random.default_rng(42)
horas = pd.date_range('2025-01-01', periods=24*120, freq='h')
h = horas.hour.values
sol = np.clip(np.sin((h-6)/12*np.pi), 0, None)            # ciclo diario del sol
nube = np.clip(rng.normal(0.3, 0.25, len(horas)), 0, 0.9)  # nubosidad
irr = 1000*sol*(1-nube)+rng.normal(0,15,len(horas))
irr = np.clip(irr, 0, None)
temp = 18 + 8*sol + rng.normal(0,1.2,len(horas))
pot = np.clip(0.18*irr*(1-0.004*(temp-25)) + rng.normal(0,2,len(horas)), 0, None)  # kW
df = pd.DataFrame({'irr':irr,'temp':temp,'nube':nube,'hora':h,'pot':pot}, index=horas)
df.head()
df['sin_h']=np.sin(2*np.pi*df['hora']/24); df['cos_h']=np.cos(2*np.pi*df['hora']/24)
cols = ['irr','temp','nube','sin_h','cos_h']
corte = int(len(df)*0.8)
Xtr, Xte = df[cols].iloc[:corte], df[cols].iloc[corte:]
ytr, yte = df['pot'].iloc[:corte], df['pot'].iloc[corte:]

## 2. Validación cruzada para series de tiempo
`TimeSeriesSplit` entrena con el pasado y valida con el bloque siguiente, avanzando por tramos. Nunca mezcla al azar.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=4)
for i,(a,b) in enumerate(tscv.split(Xtr)):
    print(f'Tramo {i+1}: entrena {len(a)} filas, valida {len(b)} filas')

## 3. Búsqueda de hiperparámetros
Probamos la profundidad y el número de árboles de un bosque aleatorio con `GridSearchCV` (sobre train, nunca sobre test).

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
rejilla = {'max_depth':[3,6,12], 'n_estimators':[50,150]}
gs = GridSearchCV(RandomForestRegressor(random_state=0, n_jobs=-1), rejilla, cv=tscv, scoring='neg_mean_absolute_error')
gs.fit(Xtr, ytr)
print('Mejores parámetros:', gs.best_params_)
print('MAE de validación :', round(-gs.best_score_,2))

## 4. Tabla comparativa
Comparamos tres modelos con MAE, RMSE y tiempo de entrenamiento. El conjunto de test se usa **una sola vez**.

In [ ]:
import time
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
modelos = {'Regresión lineal': LinearRegression(),
           'Árbol (prof. 6)': DecisionTreeRegressor(max_depth=6, random_state=0),
           'Bosque ajustado': gs.best_estimator_}
filas = []
for nombre, m in modelos.items():
    t0 = time.time(); m.fit(Xtr, ytr); seg = time.time()-t0
    p = m.predict(Xte)
    filas.append({'modelo':nombre,'MAE':round(mean_absolute_error(yte,p),2),'RMSE':round(mean_squared_error(yte,p)**0.5,2),'segundos':round(seg,2)})
comparacion = pd.DataFrame(filas).set_index('modelo')
comparacion

## 5. Explicar el modelo
Importancia de variables del bosque: qué variables pesan más. Debe coincidir con la física: la irradiancia manda.

In [ ]:
imp = pd.Series(gs.best_estimator_.feature_importances_, index=cols).sort_values()
imp.plot.barh(); plt.title('Importancia de variables'); plt.show()

## 6. Residuos
El residuo es *real − predicho*. Debe estar centrado en cero y sin patrones claros.

In [ ]:
res = yte - gs.best_estimator_.predict(Xte)
fig, ax = plt.subplots(1,2, figsize=(10,3))
ax[0].hist(res, bins=40); ax[0].set_title('Distribución del residuo')
ax[1].scatter(df.loc[yte.index,'hora'], res, s=3, alpha=0.4); ax[1].set_title('Residuo por hora')
plt.show()
print('Intervalo aproximado del 90 % del error (kW):', np.round(np.percentile(res,[5,95]),1))

## 7. Guardar el modelo (primer paso del despliegue)
Guardamos el modelo con `joblib`. Luego otra persona puede cargarlo y predecir sin reentrenar.

In [ ]:
import joblib
joblib.dump(gs.best_estimator_, 'modelo_potencia.joblib')
cargado = joblib.load('modelo_potencia.joblib')
ejemplo = pd.DataFrame([{'irr':800,'temp':27,'nube':0.1,'sin_h':np.sin(2*np.pi*12/24),'cos_h':np.cos(2*np.pi*12/24)}])
print('Predicción para el mediodía soleado (kW):', round(float(cargado.predict(ejemplo)[0]),1))

## 8. Ejercicios
1. Agregue `n_estimators` = 300 a la rejilla. ¿Mejora?
2. ¿Qué modelo elegiría si necesitara explicar la predicción a un operador?
3. Intente predecir con `irr` quitada. ¿Qué variable toma el control?
4. Describa con sus palabras qué vigilaría en producción.